In [1]:
!conda env list


# conda environments:
#
# * -> active
# + -> frozen
                         /home/njm12/ATMS_523/envs/era5-850
                         /home/njm12/ATMS_523/envs/xarray-climate
base                 *   /opt/conda



In [2]:
!conda run -p /home/njm12/ATMS_523/envs/xarray-climate python -m ipykernel install --user --name xarray-climate --display-name "Python (xarray-climate)"

Installed kernelspec xarray-climate in /home/njm12/.local/share/jupyter/kernels/xarray-climate



In [3]:
import sys
print(sys.executable)

/home/njm12/ATMS_523/envs/xarray-climate/bin/python


In [4]:
# Cell 1: Imports and environment setup
import os
import glob
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point, box
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np
import seaborn as sns
import matplotlib.patches as patches

from sklearn.cluster import DBSCAN
from geopy.distance import geodesic

#Interactive Tool Imports
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"  # or "notebook_connected"

#Statistical Significance Testing Imports
from scipy.stats import kruskal
import itertools

# Allow OGR to accept non-closed rings
os.environ["OGR_GEOMETRY_ACCEPT_UNCLOSED_RING"] = "YES"

# --- Configuration ---
base_dir = "/home/njm12/ATMS_596/ATMS-596-Capstone-Project/Land-Water"
target_crs = "EPSG:26915"  # UTM Zone 15
tornado_csv = "/home/njm12/ATMS_596/ATMS-596-Capstone-Project/CSV Files/tornadoes_with_solar_times.csv"  # Update with your CSV path
states_of_interest = ["IL", "IA", "MO"]
min_date = pd.to_datetime("1950-01-01")  # Start from first date in dataset

In [5]:
df = pd.read_csv(tornado_csv)

# ================================================================
# Basic inspection
# ================================================================

print("\nShape of dataset:")
print(df.shape)

print("\nColumn names:")
print(df.columns)

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())


Shape of dataset:
(8287, 37)

Column names:
Index(['om', 'yr', 'mo', 'dy', 'date', 'time', 'tz', 'st', 'stf', 'stn', 'mag',
       'inj', 'fat', 'loss', 'closs', 'slat', 'slon', 'elat', 'elon', 'len',
       'wid', 'ns', 'sn', 'sg', 'f1', 'f2', 'f3', 'f4', 'fc', 'datetime',
       'sunrise_local', 'sunset_local', 'day_night_bin', 'diurnal_sec',
       'nocturnal_sec', 'diurnal_hours', 'nocturnal_hours'],
      dtype='object')

First 5 rows:


,om,yr,mo,dy,date,time,tz,st,stf,stn,...,f4,fc,datetime,sunrise_local,sunset_local,day_night_bin,diurnal_sec,nocturnal_sec,diurnal_hours,nocturnal_hours
0,198,1950,12,2,1950-12-02,15:00:00,3,IL,17,7,...,0,0,1950-12-02 15:00:00-06:00,1950-12-02 07:00:56.487661-06:00,1950-12-02 16:38:03.787114-06:00,Day,34627.299453,51772.700547,9.618694,14.381306
1,199,1950,12,2,1950-12-02,16:00:00,3,IL,17,8,...,0,0,1950-12-02 16:00:00-06:00,1950-12-02 06:58:49.244195-06:00,1950-12-02 16:37:08.674669-06:00,Day,34699.430474,51700.569526,9.638731,14.361269
2,201,1950,12,2,1950-12-02,17:30:00,3,IL,17,9,...,0,0,1950-12-02 17:30:00-06:00,1950-12-02 06:57:41.974386-06:00,1950-12-02 16:39:09.003928-06:00,Night,34887.029542,51512.970458,9.690842,14.309158
3,5,1950,1,25,1950-01-25,19:30:00,3,MO,29,2,...,0,0,1950-01-25 19:30:00-06:00,1950-01-25 07:12:19.167490-06:00,1950-01-25 17:18:20.206660-06:00,Night,36361.039170,50038.960830,10.100289,13.899711
4,6,1950,1,25,1950-01-25,21:00:00,3,IL,17,3,...,0,0,1950-01-25 21:00:00-06:00,1950-01-25 07:07:14.812744-06:00,1950-01-25 16:56:38.836911-06:00,Night,35364.024167,51035.975833,9.823340,14.176660



Data types:
om                   int64
yr                   int64
mo                   int64
dy                   int64
date                object
time                object
tz                   int64
st                  object
stf                  int64
stn                  int64
mag                  int64
inj                  int64
fat                  int64
loss               float64
closs              float64
slat               float64
slon               float64
elat               float64
elon               float64
len                float64
wid                  int64
ns                   int64
sn                   int64
sg                   int64
f1                   int64
f2                   int64
f3                   int64
f4                   int64
fc                   int64
datetime            object
sunrise_local       object
sunset_local        object
day_night_bin       object
diurnal_sec        float64
nocturnal_sec      float64
diurnal_hours      float64
nocturnal_hours

In [6]:
# Ensure datetime
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)

# Sort
df = df.sort_values("datetime").reset_index(drop=True)

# ------------------------------------------------
# Parameters
# ------------------------------------------------

TIME_WINDOWS = [6, 12, 24]   # hours
SPACE_WINDOW_KM = 150        # adjust if needed
OUTBREAK_THRESHOLD = 20

# ------------------------------------------------
# Helper: build feature space (space + time)
# ------------------------------------------------

def build_feature_space(df, time_window_hours):

    df = df.copy()

    # Time in seconds
    t0 = df["datetime"].min()
    df["time_sec"] = (df["datetime"] - t0).dt.total_seconds()

    # Scale time → km-equivalent
    time_scale = SPACE_WINDOW_KM / (time_window_hours * 3600)
    df["time_scaled"] = df["time_sec"] * time_scale

    # Convert lat/lon → km (approx)
    df["x_km"] = df["slon"] * 111 * np.cos(np.radians(df["slat"]))
    df["y_km"] = df["slat"] * 111

    return df[["x_km", "y_km", "time_scaled"]].values


In [7]:
# ------------------------------------------------
# Run DBSCAN for each time window
# ------------------------------------------------

for hours in TIME_WINDOWS:

    print("\n======================================================")
    print(f"DBSCAN OUTBREAK DETECTION ({hours}-hour window)")
    print("======================================================")

    df_copy = df.copy()

    # Build feature space
    X = build_feature_space(df_copy, hours)

    # Run DBSCAN
    clustering = DBSCAN(
        eps=SPACE_WINDOW_KM,
        min_samples=3,
        metric="euclidean"
    ).fit(X)

    # Assign cluster labels
    cluster_col = f"cluster_{hours}h"
    outbreak_col = f"outbreak_{hours}h"

    df_copy[cluster_col] = clustering.labels_

    # ------------------------------------------------
    # Compute event sizes (ONLY for clusters, not noise)
    # ------------------------------------------------

    clustered = df_copy[df_copy[cluster_col] != -1]

    event_sizes = (
        clustered.groupby(cluster_col)
        .size()
        .rename("count")
    )

    # Merge event sizes back
    df_copy = df_copy.merge(
        event_sizes,
        left_on=cluster_col,
        right_index=True,
        how="left"
    )

    # ------------------------------------------------
    # CLASSIFY ALL TORNADOES (FIXED LOGIC)
    # ------------------------------------------------

    df_copy[outbreak_col] = np.where(
        df_copy[cluster_col] == -1,
        "Individual",  # noise points
        np.where(
            df_copy["count"] >= OUTBREAK_THRESHOLD,
            "Outbreak",
            "Individual"
        )
    )

    # ------------------------------------------------
    # PRINT RESULTS
    # ------------------------------------------------

    print("\nOutbreak classification counts:")
    print(df_copy[outbreak_col].value_counts())

    print("\nTotal tornadoes (sanity check):")
    print(df_copy[outbreak_col].value_counts().sum())

    print("\nTop cluster sizes:")
    print(event_sizes.sort_values(ascending=False).head(10))

    print("\nNoise points (isolated tornadoes):")
    print((df_copy[cluster_col] == -1).sum())

    # ------------------------------------------------
    # Save results back into main dataframe
    # ------------------------------------------------

    df[cluster_col] = df_copy[cluster_col]
    df[outbreak_col] = df_copy[outbreak_col]



DBSCAN OUTBREAK DETECTION (6-hour window)

Outbreak classification counts:
outbreak_6h
Individual    7088
Outbreak      1199
Name: count, dtype: int64

Total tornadoes (sanity check):
8287

Top cluster sizes:
cluster_6h
700    62
708    53
471    50
469    46
334    46
739    46
638    45
364    36
419    36
416    34
Name: count, dtype: int64

Noise points (isolated tornadoes):
3192

DBSCAN OUTBREAK DETECTION (12-hour window)

Outbreak classification counts:
outbreak_12h
Individual    7031
Outbreak      1256
Name: count, dtype: int64

Total tornadoes (sanity check):
8287

Top cluster sizes:
cluster_12h
712    62
720    53
480    51
482    50
344    46
749    46
648    45
508    37
374    36
431    36
Name: count, dtype: int64

Noise points (isolated tornadoes):
3051

DBSCAN OUTBREAK DETECTION (24-hour window)

Outbreak classification counts:
outbreak_24h
Individual    6852
Outbreak      1435
Name: count, dtype: int64

Total tornadoes (sanity check):
8287

Top cluster sizes:
cluster_2

In [8]:
# ------------------------------------------------
# Save final output
# ------------------------------------------------

output_file = "/home/njm12/ATMS_596/ATMS-596-Capstone-Project/CSV Files/tornadoes_with_dbscan_outbreaks.csv"

df.to_csv(output_file, index=False)

print("\nSaved DBSCAN results to:", output_file)


Saved DBSCAN results to: /home/njm12/ATMS_596/ATMS-596-Capstone-Project/CSV Files/tornadoes_with_dbscan_outbreaks.csv


In [9]:
# ================================================================
# USER QUERY: Inspect tornado clusters for a given date
# ================================================================

# Make sure datetime is already processed (UTC)
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)

# Choose which clustering to analyze
cluster_col = "cluster_6h"      # change to cluster_12h or cluster_24h if desired
outbreak_col = "outbreak_6h"    # matching outbreak label

# ------------------------------------------------
# User input
# ------------------------------------------------
user_date = input("Enter date (YYYY-MM-DD): ")

try:
    user_date = pd.to_datetime(user_date).date()
except:
    print("Invalid date format. Use YYYY-MM-DD.")
    raise

# ------------------------------------------------
# Filter tornadoes for that day
# ------------------------------------------------
df["date_only"] = df["datetime"].dt.date
subset = df[df["date_only"] == user_date]

print("\n======================================================")
print(f"Tornadoes on {user_date}: {len(subset)}")
print("======================================================")

if len(subset) == 0:
    print("No tornadoes found for this date.")
else:
    
    # ------------------------------------------------
    # Identify clusters present that day
    # ------------------------------------------------
    clusters_today = subset[cluster_col].unique()
    
    for cid in clusters_today:
        
        print("\n-----------------------------")
        
        if cid == -1:
            print("Cluster: NOISE (Individual tornadoes)")
            cluster_df = subset[subset[cluster_col] == -1]
        else:
            print(f"Cluster ID: {cid}")
            cluster_df = df[df[cluster_col] == cid]
        
        # ------------------------------------------------
        # Summary stats
        # ------------------------------------------------
        n = len(cluster_df)
        outbreak_flag = cluster_df[outbreak_col].iloc[0]
        
        print(f"Total tornadoes in cluster: {n}")
        print(f"Classification: {outbreak_flag}")
        
        # Time span
        start_time = cluster_df["datetime"].min()
        end_time = cluster_df["datetime"].max()
        duration = (end_time - start_time).total_seconds() / 3600
        
        print(f"Start time: {start_time}")
        print(f"End time:   {end_time}")
        print(f"Duration (hours): {duration:.2f}")
        
        # States involved
        states = cluster_df["st"].unique()
        print(f"States involved: {list(states)}")
        
        # EF-scale distribution
        ef_counts = cluster_df["mag"].value_counts().sort_index()
        print("EF-scale counts:")
        print(ef_counts.to_string())
        
        # Path length stats
        if "len" in cluster_df.columns:
            lengths_km = cluster_df["len"] * 1.60934
            print(f"Mean path length (km): {lengths_km.mean():.2f}")
            print(f"Max path length (km):  {lengths_km.max():.2f}")
        
        # Show sample rows
        print("\nSample tornadoes:")
        display(cluster_df[["date", "time", "st", "mag", "len"]].head())

print("\n======================================================")

Enter date (YYYY-MM-DD):  1974-06-20



Tornadoes on 1974-06-20: 18

-----------------------------
Cluster ID: 148
Total tornadoes in cluster: 5
Classification: Individual
Start time: 1974-06-19 23:30:00+00:00
End time:   1974-06-20 01:15:00+00:00
Duration (hours): 1.75
States involved: ['IL']
EF-scale counts:
mag
0    5
Mean path length (km): 0.16
Max path length (km):  0.16

Sample tornadoes:


,date,time,st,mag,len
1807,1974-06-19,18:30:00,IL,0,0.1
1808,1974-06-19,18:30:00,IL,0,0.1
1809,1974-06-19,18:30:00,IL,0,0.1
1810,1974-06-19,19:50:00,IL,0,0.1
1811,1974-06-19,20:15:00,IL,0,0.1



-----------------------------
Cluster ID: 149
Total tornadoes in cluster: 23
Classification: Outbreak
Start time: 1974-06-20 21:45:00+00:00
End time:   1974-06-21 01:40:00+00:00
Duration (hours): 3.92
States involved: ['IA', 'IL']
EF-scale counts:
mag
0    18
1     4
3     1
Mean path length (km): 0.99
Max path length (km):  17.06

Sample tornadoes:


,date,time,st,mag,len
1812,1974-06-20,16:45:00,IA,3,10.6
1813,1974-06-20,16:45:00,IL,0,0.1
1814,1974-06-20,17:00:00,IL,0,0.1
1815,1974-06-20,17:12:00,IL,0,0.1
1816,1974-06-20,17:30:00,IL,1,0.1


### Simple Methodology 
- ≥ 20 tornadoes → Outbreak
- < 20 tornadoes → Non-outbreak cluster
- Noise → Individual tornado

- min_samples = 3 (to detect and label cluster)

- 150 km (physically realistic outbreak scale) vs. 50 km (local clustering)

- three distinct time windows -> (6h, 12h, 24h)

- DBSCAN vs. Sequential Clustering